In [1]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/022026/Data/MEDS_MDS/data/train/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,36,2001-05-30 00:00:00,DOB,NaN
1,36,2017-04-18 18:00:00,M/R06AD02,NaN
2,36,2017-04-19 08:00:00,M/R06AD02,NaN
3,36,2017-04-19 18:00:00,M/R06AD02,NaN
4,36,2017-04-20 08:00:00,M/R06AD02,NaN
5,36,2017-04-20 18:00:00,M/R06AD02,NaN
6,36,2017-04-21 08:00:00,M/R06AD02,NaN
7,36,2017-04-21 18:00:00,M/R06AD02,NaN
8,36,2017-04-22 08:00:00,M/R06AD02,NaN
9,36,2017-04-22 18:00:00,M/R06AD02,NaN


In [2]:
len(df)

364108227

In [3]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 1774422
The patients has M-medication Codes: 1208927
The patients has D-diagnosis Codes: 1773917
The patients has P-Procedure Codes: 0
The patients has S-SKS Codes: 420586


In [4]:
subject_counts = df['subject_id'].value_counts()

In [5]:
subject_counts

921542     77725
698589     61351
2016077    58264
954669     55553
1964088    52245
           ...  
436482         2
2102646        2
636539         2
165193         2
832420         2
Name: subject_id, Length: 1774422, dtype: int64

In [6]:
s_Num = df[df['code'].str.startswith('S/', na=False)]

In [7]:
s_Num

,subject_id,time,code,numeric_value
521,360,2017-12-06 23:59:00,S/KKFD16A,NaN
979,576,2017-01-25 23:59:00,S/KCGE99,NaN
1177,576,2018-12-16 23:59:00,S/KCGE99,NaN
1534,648,2022-02-03 23:59:00,S/KPJD45,NaN
2142,864,2019-01-01 23:59:00,S/KNFJ54,NaN
...,...,...,...,...
364107298,2217717,2019-09-26 23:59:00,S/KLCA10,NaN
364107427,2217753,2018-10-01 23:59:00,S/KJFA55A,NaN
364107433,2217753,2019-10-13 23:59:00,S/KJFA15,NaN
364107452,2217789,2018-02-18 23:59:00,S/KNGD11,NaN


In [8]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('S/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('S/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only Srugery code: ", only_p_ids_to_exclude)


Number of patients with only Srugery code:  []


In [9]:
df_filtered = df[~df['code'].str.startswith('S/', na=False)]

In [10]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [11]:
subject_counts_MDS

921542     77721
698589     61351
2016077    58264
954669     55553
1964088    52243
           ...  
1714923        2
484793         2
1550293        2
67442          2
513512         2
Name: subject_id, Length: 1774422, dtype: int64

In [12]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [13]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [14]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [15]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [16]:
comparison_df

,subject_id,original_count,new_count,difference
17500,312261,2666,2637,29
54995,2162677,1314,1286,28
9314,1541863,3680,3652,28
1873,1281561,7513,7485,28
7317,121479,4130,4104,26
...,...,...,...,...
818432,512283,52,52,0
818431,592544,52,52,0
818430,309912,52,52,0
818429,512463,52,52,0


In [17]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 1353836


In [18]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [19]:
print(most_changed.head(10))


       subject_id  original_count  new_count  difference  abs_diff
17500      312261            2666       2637          29        29
9314      1541863            3680       3652          28        28
1873      1281561            7513       7485          28        28
54995     2162677            1314       1286          28        28
7317       121479            4130       4104          26        26
10071      736003            3546       3521          25        25
20636     1931799            2437       2413          24        24
22652     1755152            2313       2291          22        22
18329      189251            2601       2579          22        22
17939     1152694            2630       2609          21        21


In [20]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [21]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDS codes',
        'new_count': 'MD codes'
    }
)


In [22]:
lowest_new_count_patients

,subject_id,MDS codes,MD codes,difference,abs_diff
1684997,743588,5,3,2,2
1705749,1651508,4,3,1,1
1699807,338578,4,3,1,1
1725966,47504,4,3,1,1
1724964,816599,4,3,1,1
1726538,87919,4,3,1,1
1726593,87091,4,3,1,1
1712223,121394,4,3,1,1
1711668,545084,4,3,1,1
1706923,204548,4,3,1,1


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [23]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

1774422

In [24]:
len(df_filtered)

363494992

In [25]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('S/', na=False)].copy()
print("kept rows MDP:", len(df_filtered), " / total:", len(df))


kept rows MDP: 363494992  / total: 364108227


In [26]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
N_SHARDS = 45
df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


rows to write: 363494992


In [27]:
import numpy as np
import os

N_SHARDS = 36   #45 for Whole # 36 when we have split
OUT_DIR = "./_TrainMDP_withoutS_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 36 parquet files into ./_TrainMDP_withoutS_sharded


In [28]:
import pyarrow.parquet as pq

OUT_DIR = "./_TrainMDP_withoutS_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


0.parquet rows: 10026591
1.parquet rows: 10134318
10.parquet rows: 9777004
11.parquet rows: 10076212
12.parquet rows: 9958665
13.parquet rows: 10162382
14.parquet rows: 10107719
15.parquet rows: 10062800
16.parquet rows: 9994590
17.parquet rows: 9911632
18.parquet rows: 10171298
19.parquet rows: 10379819
2.parquet rows: 10222383
20.parquet rows: 10083224
21.parquet rows: 10192484
22.parquet rows: 10340625
23.parquet rows: 10169205
24.parquet rows: 9965616
25.parquet rows: 10116192
26.parquet rows: 10187812
27.parquet rows: 10022729
28.parquet rows: 10315881
29.parquet rows: 9884987
3.parquet rows: 9974118
30.parquet rows: 9953999
31.parquet rows: 10121189
32.parquet rows: 10238756
33.parquet rows: 10105360
34.parquet rows: 9842052
35.parquet rows: 10094384
4.parquet rows: 9877481
5.parquet rows: 10392989
6.parquet rows: 10199434
7.parquet rows: 9908729
8.parquet rows: 10296476
9.parquet rows: 10225857
TOTAL rows: 363494992


In [29]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_TrainMDP_withoutS_sharded"
DST_PREFIX = "Zahra/022026/Data/MEDS_MD/data/train"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


Local files to upload: 36
Validating arguments.
Arguments validated.
'overwrite' is set to True. Any file already present in the target will be overwritten.
Uploading files from '/mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_TrainMDP_withoutS_sharded' to 'Zahra/022026/Data/MEDS_MD/data/train'
Copying 36 files with concurrency set to 6
Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_TrainMDP_withoutS_sharded/13.parquet, file 1 out of 36. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/022026/Data/MEDS_MD/data/train/13.parquet
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/U

In [30]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])


{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Found in datastore: 36
['/0.parquet', '/1.parquet', '/10.parquet', '/11.parquet', '/12.parquet', '/13.parquet', '/14.parquet', '/15.parquet', '/16.parquet', '/17.parquet', '/18.parquet', '/19.parquet', '/2.parquet', '/20.parquet', '/21.parquet', '/22.parquet', '/23.parquet', '/24.parquet', '/25.parquet', '/26.parquet']
